In [1]:
import pandas as pd
import numpy as np
from ISLP import load_data, confusion_table
from ISLP.models import (ModelSpec as MS,
                         poly,
                         summarize)
import statsmodels.api as sm
from sklearn.model_selection import train_test_split

5. In Chapter 4, we used logistic regression to predict the probability of
default using income and balance on the Default data set. We will
now estimate the test error of this logistic regression model using the
validation set approach. Do not forget to set a random seed before
beginning your analysis.

In [2]:
Default = load_data('Default')
Default.head()

,default,student,balance,income
0,No,No,729.526495,44361.625074
1,No,Yes,817.180407,12106.134700
2,No,No,1073.549164,31767.138947
3,No,No,529.250605,35704.493935
4,No,No,785.655883,38463.495879


(a) Fit a logistic regression model that uses income and balance to
predict default.

(b) Using the validation set approach, estimate the test error of this
model. In order to do this, you must perform the following steps:

i. Split the sample set into a training set and a validation set.

ii. Fit a multiple logistic regression model using only the train
ing observations.

iii. Obtain a prediction of default status for each individual in
the validation set by computing the posterior probability of
default for that individual, and classifying the individual to
the default category if the posterior probability is greater
than 0.5.

iv. Compute the validation set error, which is the fraction of
the observations in the validation set that are misclassified

(c) Repeat the process in (b) three times, using three different splits
of the observations into a training set and a validation set. Com
ment on the results obtained.

In [3]:
X = MS(['income', 'balance']).fit_transform(Default)
y = Default['default'] == 'Yes'

In [4]:
for rdn_seed in [0,17,42]:
    print('*******************************************************\n')
    print(f'******************* Random {rdn_seed} ********************')
    X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, random_state=rdn_seed)
    modelo = sm.GLM(y_treino, X_treino, family=sm.families.Binomial())
    resultados = modelo.fit()
    print(summarize(resultados))
    predicao = resultados.predict(X_teste)>0.5
    erros = (predicao != y_teste).astype(int).sum()
    print('\nErro Total:', erros)
    print('Percentual erro:', erros/len(y_teste))
    print('\n',confusion_table(predicao, y_teste))


*******************************************************

******************* Random 0 ********************
                coef   std err       z  P>|z|
intercept -11.311000  0.502000 -22.524  0.000
income      0.000016  0.000006   2.765  0.006
balance     0.005600  0.000000  21.171  0.000

Erro Total: 72
Percentual erro: 0.0288

 Truth      False  True 
Predicted              
False       2399     67
True           5     29
*******************************************************

******************* Random 17 ********************
                coef   std err       z  P>|z|
intercept -11.811700  0.515000 -22.936    0.0
income      0.000022  0.000006   3.729    0.0
balance     0.005800  0.000000  21.703    0.0

Erro Total: 64
Percentual erro: 0.0256

 Truth      False  True 
Predicted              
False       2409     56
True           8     27
*******************************************************

******************* Random 42 ********************
                coef   std err   

A mudança de quais observações compõe os dados de treino e de teste influenciam no número de erros e coeficientes do modelo.

(d) Now consider a logistic regression model that predicts the prob
ability of default using income, balance, and a dummy variable
for student. Estimate the test error for this model using the val
idation set approach. Comment on whether or not including a
dummy variable for student leads to a reduction in the test error
rate.

In [5]:
X = MS(['income','balance','student']).fit_transform(Default)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, random_state=0)

modelo = sm.GLM(y_treino, X_treino, family=sm.families.Binomial())
resultados = modelo.fit()
print(summarize(resultados))
predicao = resultados.predict(X_teste)>0.5
erros = (predicao != y_teste).astype(int).sum()
print('\nErro Total:', erros)
print('Percentual erro:', erros/len(y_teste))
confusion_table(predicao, y_teste)

                   coef  std err       z  P>|z|
intercept    -10.587500  0.57300 -18.479  0.000
income        -0.000003  0.00001  -0.308  0.758
balance        0.005700  0.00000  21.096  0.000
student[Yes]  -0.684300  0.27800  -2.461  0.014

Erro Total: 72
Percentual erro: 0.0288


Truth,False,True
Predicted,,
False,2397,65
True,7,31


Não houve melhora na quantidade de acertos, ao adicionar o novo preditor 'student' o preditor 'income' perdeu relevância e seu p-valor mostra que o preditor se tornou irrelevante para o modelo

6. We continue to consider the use of a logistic regression model to
predict the probability of default using income and balance on the
Default data set. In particular, we will now compute estimates for the
standard errors of the income and balance logistic regression coeffi
cients in two different ways: (1) using the bootstrap, and (2) using the
standard formula for computing the standard errors in the sm.GLM()
function. Do not forget to set a random seed before beginning your
analysis.

(a) Using the summarize() and sm.GLM() functions, determine the
estimated standard errors for the coefficients associated with
income and balance in a multiple logistic regression model that
uses both predictors.

In [50]:
X = MS(['income','balance']).fit_transform(Default)
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, random_state=0)

modelo = sm.GLM(y_treino, X_treino, family=sm.families.Binomial())
resultados = modelo.fit()
print(resultados.bse)
summarize(resultados)

intercept    0.502180
income       0.000006
balance      0.000263
dtype: float64


,coef,std err,z,P>|z|
intercept,-11.311000,0.502000,-22.524,0.000
income,0.000016,0.000006,2.765,0.006
balance,0.005600,0.000000,21.171,0.000


(b) Write a function, boot_fn(), that takes as input the Default data
set as well as an index of the observations, and that outputs
the coefficient estimates for income and balance in the multiple
logistic regression model.

In [64]:
def boot_fn(dataframe: pd.DataFrame, preditores, resposta, idx_observacoes):
    dataframe = dataframe.loc[idx_observacoes]
    X = MS(preditores).fit_transform(dataframe)
    y = dataframe[resposta] == 'Yes'

    return sm.GLM(y,X, family=sm.families.Binomial()).fit().params

rdn = np.random.default_rng(42)
boot_fn(Default,['income','balance'], 'default', rdn.choice(len(Default), 10000, replace=True))

intercept   -12.207671
income        0.000030
balance       0.005869
dtype: float64

(c) Following the bootstrap example in the lab, use your boot_fn()
function to estimate the standard errors of the logistic regression
coefficients for income and balance.

In [83]:
def calc_SE(dataframe: pd.DataFrame, preditores, resposta, n=None, B=1000, seed=0):

    rdn = np.random.default_rng(seed)
    n = n or dataframe.shape[0]
    res = []
    for i in range(B):
        res.append(boot_fn(dataframe,preditores, resposta, rdn.choice(len(Default), n, replace=True)))

    return pd.DataFrame(res).std()


In [91]:
bootstrap_se = calc_SE(Default,['income','balance'], 'default')
bootstrap_se

intercept    0.435910
income       0.000005
balance      0.000231
dtype: float64

In [90]:
def calc_SE_2(dataframe: pd.DataFrame, preditores, resposta, n=None, B=1000, seed=0):

    rdn = np.random.default_rng(seed)
    n = n or dataframe.shape[0]
    primeiro_ = 0
    segundo_ = 0
    for i in range(B):
        r = boot_fn(dataframe,preditores, resposta, rdn.choice(len(Default), n, replace=True))
        primeiro_ += r
        segundo_ += r**2

    return np.sqrt(segundo_/B - (primeiro_ / B)**2)

calc_SE_2(Default,['income','balance'], 'default')

intercept    0.435692
income       0.000005
balance      0.000230
dtype: float64

(d) Comment on the estimated standard errors obtained using the
sm.GLM() function and using the bootstrap.

In [92]:
resultados.bse, bootstrap_se

(intercept    0.502180
 income       0.000006
 balance      0.000263
 dtype: float64,
 intercept    0.435910
 income       0.000005
 balance      0.000231
 dtype: float64)

In [93]:
resultados.bse - bootstrap_se

intercept    0.066270
income       0.000001
balance      0.000032
dtype: float64

Há uma leve diferença entre os valores, o calculado com o bootstrap nesse caso foi um pouco menor do que o da função GLM, isso acontece porquê o glm usa calcula utilizando a variância dos resíduos, o bootstrap calcula através de diversos valores de reamostragens, através da variabilidade das reamostragens o resultado é mais confiável